# MilaanAI — Predictions v2 (foolproof edition)

**Setup once:** Runtime → Change runtime type → **T4 GPU** → Save.

Then: Runtime → **Run all**. You'll be asked to choose files ONCE.
Select **BOTH files together** (Ctrl+click): `milaan_categorizer_lora.zip` and `test.jsonl`.

The predictions file downloads AUTOMATICALLY when done. Total ~10 min.

In [ ]:
%%capture
!pip install unsloth

In [ ]:
# Upload: select BOTH milaan_categorizer_lora.zip AND test.jsonl together
import os, shutil
from google.colab import files

print("Select BOTH files (Ctrl+click): milaan_categorizer_lora.zip and test.jsonl")
files.upload()

# Accept the zip OR loose adapter files — whatever arrived
if os.path.exists("milaan_categorizer_lora.zip"):
    os.system("unzip -o -q milaan_categorizer_lora.zip")
os.makedirs("milaan_categorizer_lora", exist_ok=True)
for f in ["adapter_config.json", "adapter_model.safetensors", "chat_template.jinja",
          "tokenizer_config.json", "tokenizer.json", "README.md"]:
    if os.path.exists(f):
        shutil.move(f, f"milaan_categorizer_lora/{f}")

missing = []
if not os.path.exists("milaan_categorizer_lora/adapter_model.safetensors"):
    missing.append("adapter zip")
if not os.path.exists("test.jsonl"):
    missing.append("test.jsonl")
while missing:
    print(f"MISSING: {missing} — upload now:")
    files.upload()
    if os.path.exists("milaan_categorizer_lora.zip"):
        os.system("unzip -o -q milaan_categorizer_lora.zip")
    missing = []
    if not os.path.exists("milaan_categorizer_lora/adapter_model.safetensors"):
        missing.append("adapter zip")
    if not os.path.exists("test.jsonl"):
        missing.append("test.jsonl")
print("ALL FILES READY")

In [ ]:
# Load model + adapter
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="milaan_categorizer_lora",
    max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True)
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"
print("model ready")

In [ ]:
# Predict -> AUTO-DOWNLOAD student_preds.jsonl when finished
import json
from tqdm import tqdm
from google.colab import files as gfiles

CATEGORIES = ['sales_receipt', 'vendor_payment', 'salary', 'rent', 'bank_charges', 'gst_tds_payment', 'loan_emi', 'utility_bill', 'upi_p2p', 'refund', 'interest_credit', 'cash_deposit', 'cash_withdrawal', 'insurance', 'other']
CATS_STR = ", ".join(CATEGORIES)
SYS = ("You classify Indian bank statement transactions into exactly one "
       "category. Reply with only the category name.")

test_rows = [json.loads(l) for l in open("test.jsonl", encoding="utf-8")]

def to_prompt(row):
    msgs = [{"role": "system", "content": SYS},
            {"role": "user", "content":
              f"Categories: {CATS_STR}\nTransaction: {row['text']}\nCategory:"}]
    return tokenizer.apply_chat_template(msgs, tokenize=False,
                                         add_generation_prompt=True)

BATCH = 32
correct = 0
with open("student_preds.jsonl", "w", encoding="utf-8") as f:
    for i in tqdm(range(0, len(test_rows), BATCH)):
        batch = test_rows[i:i+BATCH]
        inputs = tokenizer([to_prompt(r) for r in batch], return_tensors="pt",
                           padding=True, truncation=True,
                           max_length=MAX_SEQ).to("cuda")
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=8, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
        texts = tokenizer.batch_decode(out[:, inputs["input_ids"].shape[1]:],
                                       skip_special_tokens=True)
        for row, t in zip(batch, texts):
            t = t.strip().lower()
            pred = next((c for c in CATEGORIES if c in t), "other")
            f.write(json.dumps({"text": row["text"], "pred": pred}) + "\n")
            correct += (pred == row["category"])

print(f"STUDENT TEST ACCURACY: {correct/len(test_rows):.4f} "
      f"({correct}/{len(test_rows)})")
gfiles.download("student_preds.jsonl")   # <-- automatic, no extra cell
print("student_preds.jsonl downloading to your laptop now")